In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy
import random

2025-12-05 22:29:26.234746: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-05 22:29:26.234805: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-05 22:29:26.236467: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-05 22:29:26.246098: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 22:29:27.284891: W tensorflow/compiler/tf2

In [2]:
batch_size = 1024
learning_rate = 0.0001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=128, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=128, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.c = None
    def upload(self, delta_ws, delta_cs):
        for v, dw in zip(self.model.variables, delta_ws):
            v.assign_add(dw)
        for c_global, dc in zip(self.c, delta_cs):
            c_global.assign_add(dc)
        return self.model, self.c
    def download(self):
        return self.model, self.c
    def initModel(self, x):
        self.model(x)
        if self.c is None:
            self.c = [tf.Variable(tf.zeros_like(v), trainable=False) for v in self.model.trainable_variables]

In [5]:
def valiAll(index_epoch):
    m, _ = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [6]:
class Node:
    def __init__(self, id, freq):
        self.id = id
        self.freq = freq
        self.model = MLP()
        self.dataset1 = pd.read_csv('./20-24Trainset.csv', encoding='utf-8')
        self.dataset2 = pd.read_csv('./50-54Trainset.csv', encoding='utf-8')
        self.dataset = pd.concat([self.dataset1, self.dataset2], axis=0).sample(frac=1).reset_index(drop=True)
        self.dataset = self.dataset[self.dataset['freq'].isin(self.freq)]
        self.X = self.dataset.loc[:,'freq':'L2'].to_numpy(dtype = np.float32)
        self.y = self.dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=self.X.shape[0])
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
        self.c_i = None
    def train(self, idx_epoch):
        # 1. 从服务器下载当前全局模型参数 w_t 和全局控制变量 c
        global_model, c_global = ps.download()
        self.model = copy.deepcopy(global_model)

        # 初始化本地 c_i
        if self.c_i is None:
            self.c_i = [tf.zeros_like(v) for v in self.model.variables]

        # 2. 保存本轮开始时的参数 w_t
        w_before = [tf.identity(v) for v in self.model.variables]

        step = 0
        last_tr_mse = None
        last_tr_rmse = None
        last_tr_mae = None
        last_tr_r2 = None

        # 3. 本地用修正梯度做 K 步更新
        for X, y in self.dataset_train:

            with tf.GradientTape() as tape:
                y_pred = self.model(X)
                tr_mse = tf.reduce_mean(tf.square(y_pred - y))

            grads = tape.gradient(tr_mse, self.model.variables)

            # 修正梯度：g - c_i + c
            corrected_grads = [
                g - ci + cg
                for g, ci, cg in zip(grads, self.c_i, c_global)
            ]

            # 用简单 SGD 更新本地模型参数
            for v, g_corr in zip(self.model.variables, corrected_grads):
                v.assign_sub(learning_rate * g_corr)

            # 记录最后一个 batch 的指标（方便打印）
            last_tr_mse = tr_mse
            last_tr_rmse = tf.sqrt(tr_mse)
            last_tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
            last_tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(
                tf.square(y - tf.reduce_mean(y))
            )
            step += 1
        # 4. 本地更新完成后，计算 Δw_i
        w_after = [tf.identity(v) for v in self.model.variables]
        delta_w = [
            w_a - w_b
            for w_a, w_b in zip(w_after, w_before)
        ]

        # 5. 按 SCAFFOLD 公式更新本地控制变量 c_i，并得到 Δc_i
        #    c_i_new = c_i - c + (1 / (K * η)) * (w_before - w_after)
        K = max(step, 1)  # 防止除零
        scale = 1.0 / (K * learning_rate)

        new_c_i = []
        delta_c_i = []
        for ci_old, c_g, w_b, w_a in zip(self.c_i, c_global, w_before, w_after):
            ci_new = ci_old - c_g + scale * (w_b - w_a)
            new_c_i.append(ci_new)
            delta_c_i.append(ci_new - ci_old)

        # 更新本地 c_i
        self.c_i = new_c_i

        # 6. 把 Δw_i 和 Δc_i 上传给服务器
        global_model, c_global = ps.upload(delta_w, delta_c_i)

        # 7. 打印训练信息 + 记录 r2 + 调用你的验证函数
        if last_tr_mse is not None:
            print("node:{} round:{}".format(self.freq, idx_epoch))
            print("train mse:{} rmse:{} mae:{} r2:{}".format(
                last_tr_mse, last_tr_rmse, last_tr_mae, last_tr_r2
            ))
            r2s[self.id][idx_epoch] = last_tr_r2.numpy()

In [7]:
r2s = {0:{}, 1:{}, 2:{}, 3:{}, 4:{}}
r2sv = []

In [8]:
test_dataset = pd.read_csv("testset.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L2'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-05 22:29:28.439458: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21505 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [10]:
nodeList = [Node(0, [2.0, 5.0]), Node(1, [2.1, 5.1]), Node(2, [2.2, 5.2]), Node(3, [2.3, 5.3]), Node(4, [2.4, 5.4])]

In [11]:
orders = [0, 1, 2, 3, 4]
turn = [np.array([[92, 146], [158, 255], [347, 475], [531, 555]]), 
        np.array([[42, 116], [226, 277], [363, 423], [543, 600]]), 
        np.array([[0, 200], [214, 252], [271, 347], [474, 528]]),
        np.array([[68, 132], [173, 305], [418, 423], [563, 587]]),
        np.array([[7, 151], [216, 305], [357, 360], [420, 538]]),]
for i in range(600):
    random.shuffle(orders)
    for j in orders:
        for l, r in turn[j]:
            if l <= i < r:
                nodeList[j].train(i)
    valiAll(i)

node:[2.2, 5.2] round:0
train mse:0.21159784495830536 rmse:0.45999765396118164 mae:0.36935001611709595 r2:-0.7345389127731323
mse:0.2078075259923935 rmse:0.45585909485816956 mae:0.3668803870677948 r2:-0.7020035982131958
node:[2.2, 5.2] round:1
train mse:0.15094609558582306 rmse:0.388517826795578 mae:0.30668848752975464 r2:-0.23806631565093994
mse:0.15212275087833405 rmse:0.3900291621685028 mae:0.30986207723617554 r2:-0.24592936038970947
node:[2.2, 5.2] round:2
train mse:0.13454917073249817 rmse:0.3668094575405121 mae:0.2934982478618622 r2:-0.10233139991760254
mse:0.13572143018245697 rmse:0.3684038817882538 mae:0.2917768359184265 r2:-0.11159765720367432
node:[2.2, 5.2] round:3
train mse:0.12789854407310486 rmse:0.35762906074523926 mae:0.2832711637020111 r2:-0.04700446128845215
mse:0.1285005360841751 rmse:0.35846972465515137 mae:0.2840716242790222 r2:-0.05245649814605713
node:[2.2, 5.2] round:4
train mse:0.1228167936205864 rmse:0.3504522740840912 mae:0.2776772081851959 r2:-0.005852699279

In [12]:
for v in r2sv:
    print(v)

-0.7020036
-0.24592936
-0.11159766
-0.0524565
-0.01641202
0.0100348
0.031150877
0.087926745
0.119693816
0.13844079
0.15164661
0.16207749
0.17068225
0.17801231
0.18448853
0.19023794
0.19556165
0.20028758
0.2047844
0.20891243
0.21278632
0.21647024
0.21990615
0.2231592
0.22628611
0.22926271
0.2320801
0.23484409
0.23738009
0.23993707
0.24237424
0.24472016
0.24697602
0.24920744
0.2513231
0.2533986
0.25538063
0.25741112
0.25934476
0.26124322
0.26310444
0.26488483
0.26965255
0.2739386
0.27773964
0.2814557
0.2849452
0.2883857
0.2916429
0.294828
0.29791075
0.30089706
0.3037868
0.30660427
0.3093211
0.31197387
0.31454486
0.31704855
0.319463
0.3218233
0.32409018
0.3263176
0.3284657
0.33056378
0.33256
0.33454448
0.33643425
0.3382948
0.34061998
0.34393966
0.34673846
0.34920466
0.35182148
0.35390168
0.3564248
0.35842937
0.36048007
0.3622933
0.36404926
0.3658132
0.36740226
0.3689745
0.3703907
0.37178707
0.3731699
0.374457
0.37562287
0.37685633
0.3780334
0.37905544
0.3801943
0.3811295
0.38221908
0.3848